In [ ]:
import os
import json
import openslide
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm
from glob import glob
def create_directory(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
json_list = glob('../../data/Leica/registration_params/*_HR_*.json')
print(f"총 {len(json_list)}개의 슬라이드 쌍 발견")

# 출력 폴더 설정
output_base = '../../data/SS_virtual_stain_data/menbrane'
create_directory(output_base)
folder_pdl1_mpp05 = os.path.join(output_base, 'ihc_mpp05')      # PD-L1 mpp 0.5
folder_he_mpp05= os.path.join(output_base, 'he_mpp05')          # HE mpp 0.5 (좌표변환)


create_directory(folder_pdl1_mpp05)
create_directory(folder_he_mpp05)


print(f"출력 폴더:")
print(f"  - PD-L1 mpp 0.5: {folder_pdl1_mpp05}")
print(f"  - HE mpp 0.5:    {folder_he_mpp05}")

In [ ]:
# 썸네일 및 마스크 생성 함수
def get_thumbnail_and_mask(slide_path, downsample=30):
    slide = openslide.OpenSlide(slide_path)
    thumb = slide.get_thumbnail((slide.dimensions[0]//downsample, slide.dimensions[1]//downsample))
    thumb_np = np.array(thumb)
    gray = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return slide, thumb_np, mask


def _get_dim(params, *keys):
    """JSON 키가 여러 이름(pdl1_*/ihc_*)으로 저장될 수 있어 순서대로 시도"""
    d = params['dimensions']
    for k in keys:
        if k in d:
            return d[k]
    raise KeyError(f"None of {keys} found in dimensions: {list(d.keys())}")


def apply_transformation_to_mask(mask, thumb_shape, params):
    """HE 마스크를 IHC 공간으로 변환"""
    from scipy.ndimage import rotate as scipy_rotate

    angle = params['transformation']['thumbnail']['angle_degrees']
    scale = params['transformation']['thumbnail']['scale']
    tx = int(params['transformation']['thumbnail']['translation_x'])
    ty = int(params['transformation']['thumbnail']['translation_y'])

    rotated_mask = scipy_rotate(mask, angle, reshape=False, order=0)
    new_h, new_w = int(mask.shape[0] * scale), int(mask.shape[1] * scale)
    scaled_mask = cv2.resize(rotated_mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    # ihc_thumbnail 또는 pdl1_thumbnail 키 모두 허용
    ihc_thumb_dim = _get_dim(params, 'ihc_thumbnail', 'pdl1_thumbnail')
    target_h = ihc_thumb_dim['height']
    target_w = ihc_thumb_dim['width']
    result_mask = np.zeros((target_h, target_w), dtype=np.uint8)

    start_y = (target_h - scaled_mask.shape[0]) // 2 + ty
    start_x = (target_w - scaled_mask.shape[1]) // 2 + tx
    src_sy = max(0, -start_y); src_sx = max(0, -start_x)
    src_ey = min(scaled_mask.shape[0], target_h - start_y)
    src_ex = min(scaled_mask.shape[1], target_w - start_x)
    dst_sy = max(0, start_y); dst_sx = max(0, start_x)
    dst_ey = dst_sy + (src_ey - src_sy); dst_ex = dst_sx + (src_ex - src_sx)
    if src_ey > src_sy and src_ex > src_sx:
        result_mask[dst_sy:dst_ey, dst_sx:dst_ex] = scaled_mask[src_sy:src_ey, src_sx:src_ex]
    return result_mask


def transform_point_pdl1_to_he(ihc_x, ihc_y, params):
    """IHC WSI 좌표를 HE WSI 좌표로 역변환
    순방향(get_transform_matrix):
      ihc_x = cos_a*(he_x-he_cx) - sin_a*(he_y-he_cy) + he_cx + tx
      ihc_y = sin_a*(he_x-he_cx) + cos_a*(he_y-he_cy) + he_cy + ty
    역변환: HE center 기준으로 빼고, 회전 부호 반전
    """
    angle = params['transformation']['fullres']['angle_degrees']
    scale = params['transformation']['fullres']['scale']
    tx = params['transformation']['fullres']['translation_x']
    ty = params['transformation']['fullres']['translation_y']

    he_dim = _get_dim(params, 'he_full')
    he_cx  = he_dim['width']  / 2
    he_cy  = he_dim['height'] / 2

    angle_rad = np.radians(angle)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)

    # 순방향 translation 제거 (HE center 기준), scale 역변환
    u = (ihc_x - he_cx - tx) / scale
    v = (ihc_y - he_cy - ty) / scale

    # 역회전 (부호 반전)
    he_x = cos_a * u + sin_a * v + he_cx
    he_y = -sin_a * u + cos_a * v + he_cy

    return he_x, he_y

In [ ]:
def extract_patch(slide, center_x, center_y, patch_size_wsi, patch_resize=2048, level=0):
    """
    WSI에서 중점 기준으로 패치 추출
    
    Args:
        slide: OpenSlide 객체
        center_x, center_y: 패치 중점 좌표 (level 0 기준)
        patch_size_wsi: WSI에서 추출할 패치 크기 (level 0 픽셀)
        patch_resize: 최종 출력 패치 크기 (픽셀)
        level: 추출할 레벨
    
    Returns:
        patch: numpy array (RGB), 또는 None (유효하지 않은 경우)
    """
    x = int(center_x - patch_size_wsi / 2)
    y = int(center_y - patch_size_wsi / 2)
    
    if x < 0 or y < 0 or x + patch_size_wsi > slide.dimensions[0] or y + patch_size_wsi > slide.dimensions[1]:
        return None
    
    patch = slide.read_region((x, y), level, (patch_size_wsi, patch_size_wsi))
    patch = np.array(patch.convert('RGB'))
    
    if patch_resize != patch_size_wsi:
        patch = cv2.resize(patch, (patch_resize, patch_resize), interpolation=cv2.INTER_LINEAR)
    
    return patch


def get_valid_patch_positions(common_mask, pdl1_slide, pdl1_mpp,
                               target_mpp=0.5, patch_size=2048,
                               overlap=0.5, threshold=0.3):
    """
    유효한 패치 위치 계산
    
    Args:
        common_mask: HE 변환 마스크 AND PD-L1 마스크
        pdl1_slide: PD-L1 슬라이드 객체
        pdl1_mpp: PD-L1 슬라이드의 mpp
        target_mpp: 추출 기준 mpp (기본 0.5)
        patch_size: 출력 패치 크기 (기본 2048)
        overlap: 패치 겹침 비율, 0.5 = 50% (기본 0.5)
        threshold: 유효 조직 비율 임계값 (기본 0.3)
    
    Returns:
        valid_positions: [(center_x, center_y), ...] WSI level-0 좌표 리스트
    """
    # WSI 레벨0 기준 패치 크기 (읽어야 할 픽셀 수)
    patch_size_wsi = int(patch_size * (target_mpp / pdl1_mpp))
    # 50% overlap → stride = patch_size_wsi * (1 - overlap)
    stride_wsi = int(patch_size_wsi * (1.0 - overlap))
    
    wsi_width, wsi_height = pdl1_slide.dimensions
    thumb_scale_x = common_mask.shape[1] / wsi_width
    thumb_scale_y = common_mask.shape[0] / wsi_height
    
    # 썸네일 기준 패치/스트라이드 크기
    patch_thumb_w = int(patch_size_wsi * thumb_scale_x)
    patch_thumb_h = int(patch_size_wsi * thumb_scale_y)
    half_w = patch_thumb_w // 2
    half_h = patch_thumb_h // 2
    
    valid_positions = []
    
    cy_list = range(patch_size_wsi // 2, wsi_height - patch_size_wsi // 2, stride_wsi)
    cx_list = range(patch_size_wsi // 2, wsi_width  - patch_size_wsi // 2, stride_wsi)
    
    for center_y in cy_list:
        for center_x in cx_list:
            thumb_cx = int(center_x * thumb_scale_x)
            thumb_cy = int(center_y * thumb_scale_y)
            
            y1 = max(0, thumb_cy - half_h)
            y2 = min(common_mask.shape[0], thumb_cy + half_h)
            x1 = max(0, thumb_cx - half_w)
            x2 = min(common_mask.shape[1], thumb_cx + half_w)
            
            if y2 <= y1 or x2 <= x1:
                continue
            
            patch_mask = common_mask[y1:y2, x1:x2]
            valid_ratio = np.sum(patch_mask > 0) / patch_mask.size
            
            if valid_ratio >= threshold:
                valid_positions.append((center_x, center_y))
    
    return valid_positions, patch_size_wsi

In [ ]:
# 메인 패치 추출 루프
TARGET_MPP     = 0.5    # 추출 기준 mpp
PATCH_SIZE     = 2048   # 출력 패치 크기 (픽셀)
OVERLAP        = 0.2    # 20% 겹침
VALID_THRESHOLD = 0.3
DOWNSAMPLE     = 30

total_patches = 0

for json_idx, json_path in enumerate(tqdm(json_list, desc="Processing slides")):
    with open(json_path, 'r') as f:
        json_data = json.load(f)
    
    slide_name = os.path.basename(json_path).replace('.json', '')
    print(f"\n[{json_idx+1}/{len(json_list)}] Processing: {slide_name}")
    
    try:
        he_slide   = openslide.OpenSlide(json_data['files']['he_slide'])
        pdl1_slide = openslide.OpenSlide(json_data['files']['ihc_slide'])
    except Exception as e:
        print(f"  Error opening slides: {e}")
        continue
    
    # MPP 정보
    pdl1_mpp = float(pdl1_slide.properties.get('openslide.mpp-x', 0.25))
    he_mpp   = float(he_slide.properties.get('openslide.mpp-x', 0.25))
    print(f"  MPP: PD-L1={pdl1_mpp}, HE={he_mpp}")
    
    # 마스크 생성
    _, he_thumb,   he_mask   = get_thumbnail_and_mask(json_data['files']['he_slide'],   DOWNSAMPLE)
    _, pdl1_thumb, pdl1_mask = get_thumbnail_and_mask(json_data['files']['ihc_slide'], DOWNSAMPLE)
    
    # HE 마스크 → PD-L1 공간으로 변환
    he_transformed_mask = apply_transformation_to_mask(he_mask, pdl1_thumb.shape, json_data)
    
    # 공통 마스크 (HE 변환 AND PD-L1)
    common_mask = np.logical_and(he_transformed_mask > 0, pdl1_mask > 0).astype(np.uint8) * 255
    
    # 유효 패치 위치 계산 (mpp 0.5, 2048px, 50% overlap)
    valid_positions, patch_size_wsi = get_valid_patch_positions(
        common_mask, pdl1_slide, pdl1_mpp,
        target_mpp=TARGET_MPP, patch_size=PATCH_SIZE,
        overlap=OVERLAP, threshold=VALID_THRESHOLD
    )
    # HE도 동일 mpp 기준으로 WSI 픽셀 크기 계산
    he_patch_size_wsi = int(PATCH_SIZE * (TARGET_MPP / he_mpp))
    
    print(f"  WSI patch size: PD-L1={patch_size_wsi}px, HE={he_patch_size_wsi}px (at mpp {TARGET_MPP})")
    print(f"  Stride: {int(patch_size_wsi * (1 - OVERLAP))}px ({int(OVERLAP*100)}% overlap)")
    print(f"  Valid positions: {len(valid_positions)}")
    
    # 패치 추출 및 저장
    patch_count = 0
    for pos_idx, (cx, cy) in enumerate(tqdm(valid_positions, desc="  Extracting patches", leave=False)):
        patch_name = f"{slide_name}_{pos_idx:05d}_x{cx}_y{cy}.png"
        
        # PD-L1 패치 (mpp 0.5, 2048px)
        pdl1_patch = extract_patch(pdl1_slide, cx, cy, patch_size_wsi, PATCH_SIZE)
        if pdl1_patch is None:
            continue
        
        # HE 패치 (좌표 역변환 후 추출)
        he_cx, he_cy = transform_point_pdl1_to_he(cx, cy, json_data)
        he_patch = extract_patch(he_slide, he_cx, he_cy, he_patch_size_wsi, PATCH_SIZE)
        if he_patch is None:
            continue
        
        Image.fromarray(pdl1_patch).save(os.path.join(folder_pdl1_mpp05, patch_name))
        Image.fromarray(he_patch).save(os.path.join(folder_he_mpp05,   patch_name))
        patch_count += 1
    
    print(f"  Saved: {patch_count} patches")
    total_patches += patch_count
    
    he_slide.close()
    pdl1_slide.close()

print(f"\n=== 완료 ===")
print(f"총 추출된 패치 수: {total_patches}")
print(f"저장 위치:")
print(f"  - PD-L1 mpp {TARGET_MPP}: {folder_pdl1_mpp05}")
print(f"  - HE mpp {TARGET_MPP}:    {folder_he_mpp05}")